DataSet

In [1]:
connexions = [
    {"id": 1, "type": "normal",  "port": "habituel",   "duree": "courte"},
    {"id": 2, "type": "attaque", "port": "inhabituel", "duree": "longue"},
    {"id": 3, "type": "normal",  "port": "habituel",   "duree": "courte"},
    {"id": 4, "type": "normal",  "port": "habituel",   "duree": "longue"},
    {"id": 5, "type": "attaque", "port": "inhabituel", "duree": "longue"},
    {"id": 6, "type": "normal",  "port": "habituel",   "duree": "courte"},
    {"id": 7, "type": "normal",  "port": "inhabituel", "duree": "courte"},
    {"id": 8, "type": "normal",  "port": "habituel",   "duree": "courte"},
    {"id": 9, "type": "attaque", "port": "inhabituel", "duree": "longue"},
    {"id": 10, "type": "normal", "port": "habituel",   "duree": "courte"},
]

In [2]:
nombre_attaque=sum(1 for c in connexions if c["type"] == "attaque")
print("Nombre d'attaque :", nombre_attaque)
nombre_normal=sum(1 for c in connexions if c["type"] == "normal")
print("Nombre de connexions normales :", nombre_normal)
nombre_durations_longue=sum(1 for c in connexions if c["duree"] == "longue")
print("Nombre de connexions de durée longue :", nombre_durations_longue)
nombre_totale=len(connexions)
print("Nombre total de connexions :", nombre_totale)

attaques=[c for c in connexions if c["type"] == "attaque"]
nombre_durations_longue_parmi_attaque=sum(1 for c in attaques if c["duree"]=="longue" )
probabilite_durations_longue_parmi_attaque=nombre_durations_longue_parmi_attaque/nombre_attaque
print("Probabilité que la durée d'une connexion d'attaque soit longue :", probabilite_durations_longue_parmi_attaque)

normal=[c for c in connexions if c["type"] == "normal"]
nombre_durations_longue_parmi_normal=sum(1 for c in normal if c["duree"]=="longue" )
probabilite_durations_longue_parmi_normal=nombre_durations_longue_parmi_normal/nombre_normal
print("Probabilité que la durée d'une connexion normale soit longue :", probabilite_durations_longue_parmi_normal)



Nombre d'attaque : 3
Nombre de connexions normales : 7
Nombre de connexions de durée longue : 4
Nombre total de connexions : 10
Probabilité que la durée d'une connexion d'attaque soit longue : 1.0
Probabilité que la durée d'une connexion normale soit longue : 0.14285714285714285


In [3]:
# Probabilités déjà calculées
proba_attaque = 3 / 10           # P(attaque)
proba_normal = 7 / 10            # P(normal)

proba_inhabituel_sachant_attaque = 1.0      # calculé avant
proba_inhabituel_sachant_normal = 1 / 7     # à toi de vérifier si ça correspond à ton calcul équivalent pour le port

proba_longue_sachant_attaque = 1.0          # ton résultat
proba_longue_sachant_normal = 1 / 7         # ton résultat

# Nouvelle connexion à classifier : port inhabituel + durée longue
# Score pour "attaque" (on multiplie les probabilités conditionnelles, en supposant l'indépendance)
score_attaque = proba_inhabituel_sachant_attaque * proba_longue_sachant_attaque * proba_attaque

# Score pour "normal"
score_normal = proba_inhabituel_sachant_normal * proba_longue_sachant_normal * proba_normal

print("Score 'attaque' :", score_attaque)
print("Score 'normal' :", score_normal)

if score_attaque > score_normal:
    print("\n→ Prédiction : ATTAQUE")
else:
    print("\n→ Prédiction : NORMAL")

Score 'attaque' : 0.3
Score 'normal' : 0.014285714285714284

→ Prédiction : ATTAQUE


In [4]:
def entrainer_naive_bayes(connexions,features, cible):
  classes=list(set(c[cible] for c in connexions))
  total=len(connexions)

  proba_classes={}
  for classe in classes:
    nombre=sum(1 for c in connexions if c[cible]==classe)
    proba_classes[classe]=nombre/total
  proba_conditionnelles={}
  for classe in classes:
    connexions_classe=[c for c in connexions if c[cible]== classe]
    proba_conditionnelles[classe]={}
    for feature in features:
      valeurs=list(set(c[feature] for c in connexions))
      for valeur in valeurs:
        nombre=sum(1 for c in connexions_classe if c[feature]==valeur)
        proba_conditionnelles[classe][valeur]=nombre/len(connexions_classe)
  return proba_classes,proba_conditionnelles

def predire(nouvelle_connexion, features, proba_classes, proba_conditionnelles):
  scores={}
  for classe in proba_classes:
    score=proba_classes[classe]
    for feature in features:
      valeur= nouvelle_connexion[feature]
      cle=f"{feature}={valeur}"
      score *= proba_conditionnelles[classe].get(cle,1e-9)
      scores[classe]=score
    prediction=max(scores,key=lambda x: scores[x])
    return prediction ,scores


In [5]:
features = ["port", "duree"]
proba_classes, proba_conditionnelles = entrainer_naive_bayes(connexions, features, "type")

print("P(classe) :", proba_classes)
print("\nProbabilités conditionnelles :", proba_conditionnelles)

# Test avec notre nouvelle connexion
nouvelle = {"port": "inhabituel", "duree": "longue"}
prediction, scores = predire(nouvelle, features, proba_classes, proba_conditionnelles)

print(f"\nNouvelle connexion : {nouvelle}")
print(f"Scores : {scores}")
print(f"→ Prédiction : {prediction.upper()}")

P(classe) : {'normal': 0.7, 'attaque': 0.3}

Probabilités conditionnelles : {'normal': {'inhabituel': 0.14285714285714285, 'habituel': 0.8571428571428571, 'courte': 0.8571428571428571, 'longue': 0.14285714285714285}, 'attaque': {'inhabituel': 1.0, 'habituel': 0.0, 'courte': 0.0, 'longue': 1.0}}

Nouvelle connexion : {'port': 'inhabituel', 'duree': 'longue'}
Scores : {'normal': 7e-19}
→ Prédiction : NORMAL
